<a href="https://colab.research.google.com/github/NatalieAleksandrova2026/Analysis_Cafe_Sales/blob/main/EDA_Cafe_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analysis cafe sales  
*Cafe Sales - Dirty Data for Cleaning Training*  
  
Dirty Cafe Sales Dataset  
**Overview**  
The Dirty Cafe Sales dataset contains 10,000 rows of synthetic data representing sales transactions in a cafe. This dataset is intentionally "dirty," with missing values, inconsistent data, and errors introduced to provide a realistic scenario for data cleaning and exploratory data analysis (EDA). It can be used to practice cleaning techniques, data wrangling, and feature engineering.

**File Information**  
* File Name: dirty_cafe_sales.csv  
* Number of Rows: 10,000  
* Number of Columns: 8    
- `Transaction ID`	A unique identifier for each transaction. Always present and unique.  
- `Item`	The name of the item purchased. May contain missing or invalid values (e.g., "ERROR").  
- `Quantity`	The quantity of the item purchased. May contain missing or invalid values.    
- `Price Per Unit`	The price of a single unit of the item. May contain missing or invalid values.  
- `Total Spent`	The total amount spent on the transaction. Calculated as Quantity * Price Per Unit.  
- `Payment Method`	The method of payment used. May contain missing or invalid values (e.g., None, "UNKNOWN").  
- `Location`	The location where the transaction occurred. May contain missing or invalid values.	  
- `Transaction Date`	The date of the transaction. May contain missing or incorrect values.


`!pip install numpy, pandas, matplotlib`

In [203]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


##Downloading dataset

In [204]:
url = 'https://raw.githubusercontent.com/NatalieAleksandrova2026/Analysis_Cafe_Sales/refs/heads/main/data/dirty_cafe_sales.csv'

df_origin = pd.read_csv(url)
df_origin.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [205]:
df = df_origin.copy()

##Primary data diagnosis

In [206]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [207]:
print('Data types (raw):\n')
df.dtypes

Data types (raw):



,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


Problem: Numeric columns (Quantity	object,
Price Per Unit	object, Total Spent)  
Transaction Date	object

In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


Missing values

In [209]:
print('Missing values:\n')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_pct
missing_report = pd.DataFrame({
    'count': missing,
    '%': missing_pct
})
missing_report[missing_report['count'] > 0]

Missing values:



,count,%
Item,333,3.3
Quantity,138,1.4
Price Per Unit,179,1.8
Total Spent,173,1.7
Payment Method,2579,25.8
Location,3265,32.6
Transaction Date,159,1.6


In [210]:
print('Unique values:\n')
print('---Item---\n')
print(df['Item'].value_counts(dropna=False))


print('---Payment Method---\n')
display(df['Payment Method'].value_counts(dropna=False))
print('---Location---\n')
print(df['Location'].value_counts(dropna=False))

Unique values:

---Item---

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64
---Payment Method---



,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


---Location---

Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


In [211]:
print("Find lines with'ERROR' or 'UNKNOWN'\n")
# df.isin(['ERROR', 'UNKNOWN']).any()
display(df.isin(['ERROR', 'UNKNOWN']).sum())
mask_error = df.isin(['ERROR', 'UNKNOWN']).any(axis=1).sum()
print(f'Lines with Error or Unknown: {mask_error.sum()}')

Find lines with'ERROR' or 'UNKNOWN'



,0
Transaction ID,0
Item,636
Quantity,341
Price Per Unit,354
Total Spent,329
Payment Method,599
Location,696
Transaction Date,301


Lines with Error or Unknown: 2845


## Data Cleaning

`df_origin` - the original dataset  
`df` - the copy

In [212]:
print('Rename columns:\n')
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'Transaction ID': 'transaction_id',
    'Item': 'item',
    'Quantity': 'quantity',
    'Price Per Unit': 'price_per_unit',
    'Total Spent': 'total_spent',
    'Payment Method': 'payment_method',
    'Location': 'location',
    'Transaction Date': 'transaction_date'
})
df


Rename columns:



,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [213]:
print('Rename columns:\n')
df.columns = df.columns.str.strip()
print(list(df.columns))

Rename columns:

['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


In [214]:
'   Transaction Date  '.strip().lower()

'transaction date'

In [215]:
df.columns.str.lower().str.split()

Index([  ['transaction_id'],             ['item'],         ['quantity'],
         ['price_per_unit'],      ['total_spent'],   ['payment_method'],
               ['location'], ['transaction_date']],
      dtype='object')

In [216]:
"_".join("     Transaction   Date   ".lower().split())

'transaction_date'

In [217]:
df.columns.str.lower().str.replace(" ", "_")

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [218]:
# ERROR / UNKNOWN change to NaN
df.replace(['ERROR', 'UNKNOWN', 'error', 'unknown', ''], np.nan, inplace=True)
df

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [219]:
#Converting numeric columns
for col in ['quantity', 'price_per_unit', 'total_spent']:
  df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes


,0
transaction_id,object
item,object
quantity,float64
price_per_unit,float64
total_spent,float64
payment_method,object
location,object
transaction_date,object


new_df = pd.DataFrame({"quantity": [0, 3, 5, '', np.nan, "q"]})
` display(new_df)`
` new_df.info()`
pd.to_numeric(new_df['quantity'], errors='coerce')

In [220]:
#Date Convertion
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df.dtypes
df['transaction_date']


,transaction_date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11
...,...
9995,2023-08-30
9996,2023-06-02
9997,2023-03-02
9998,2023-12-02


In [221]:
print('Converting columns:\n')
df.dtypes

Converting columns:



,0
transaction_id,object
item,object
quantity,float64
price_per_unit,float64
total_spent,float64
payment_method,object
location,object
transaction_date,datetime64[ns]


In [222]:
#Extracting time signatures
df['year'] = df['transaction_date'].dt.year.astype('Int64')
df.year
df['month'] = df['transaction_date'].dt.month.astype('Int64')
df.month
# df['month_name'] = df['transaction_name'].dt.month_name()
df['month_name'] = df['transaction_date'].dt.strftime('%B')
df.month_name
df['weekday'] = df['transaction_date'].dt.day_name()
df['week'] = df['transaction_date'].dt.isocalendar().week.astype('Int64')
df['week']
# display(df[['year', 'month', 'month_name', 'weekday', 'week']].dtypes)
df[['year', 'month', 'month_name', 'weekday', 'week']].head()


,year,month,month_name,weekday,week
0,2023,9,September,Friday,36
1,2023,5,May,Tuesday,20
2,2023,7,July,Wednesday,29
3,2023,4,April,Thursday,17
4,2023,6,June,Sunday,23


In [223]:
#Restoring total_spent (quantity * price)

mask_recoverable = (
    df['total_spent'].isna() &
    df['quantity'].notna() &
    df['price_per_unit'].notna()
)

df.loc[mask_recoverable, 'total_spent'] = (
    df.loc[mask_recoverable, 'quantity'] *
    df.loc[mask_recoverable, 'price_per_unit']
)
print(f'Restoring total_spent: {mask_recoverable.sum()} string (quantity * price)')
df

Restoring total_spent: 462 string (quantity * price)


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,year,month,month_name,weekday,week
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,9,September,Friday,36
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,2023,5,May,Tuesday,20
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,2023,7,July,Wednesday,29
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27,2023,4,April,Thursday,17
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,6,June,Sunday,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,NaN,2023-08-30,2023,8,August,Wednesday,35
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02,2023,6,June,Friday,22
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02,2023,3,March,Thursday,9
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital Wallet,NaN,2023-12-02,2023,12,December,Saturday,48


In [224]:
(df['total_spent'].notna() & df['quantity'].notna() & df['price_per_unit'].isna() ).sum()

np.int64(495)

In [225]:
# dict of median prices
price_map = (
    df.dropna(subset=['item', 'price_per_unit'])
    .groupby('item')['price_per_unit']
    .median()
    .to_dict()
)

mask_price = df['price_per_unit'].isna() & df['item'].notna()
df.loc[mask_price, 'price_per_unit'] = df.loc[mask_price, 'item'].map(price_map)
df[58:59]
# mask_price.sum()
print(f'Restoring price_per_unit: {mask_price.sum()} string (median by items)')

Restoring price_per_unit: 479 string (median by items)


In [226]:
# type(df[mask_price])  # DataFrame
# type(df[mask_price]['item'])  # Series
# type(df[mask_price][['item', 'price_per_unit']])  # DataFrame
# type(df.loc[mask_price, 'item'])  # Series
# type(df.loc[mask_price, ['item', 'price_per_unit']])  # DataFrame

In [227]:
#Restoring total_spent (new_price - median_item)

mask_recoverable2 = (
    df['total_spent'].isna() &
    df['quantity'].notna() &
    df['price_per_unit'].notna()
)

df.loc[mask_recoverable2, 'total_spent'] = (
    df.loc[mask_recoverable2, 'quantity'] *
    df.loc[mask_recoverable2, 'price_per_unit']
)
print(f'Restoring total_spent: {mask_recoverable2.sum()} string (new price - median item)')


Restoring total_spent: 17 string (new price - median item)


In [228]:
# fillna = mode
mode_item = df['item'].mode()[0]
# df['item'].fillna(mode_item, inplace=True) old version
df.fillna({'item': mode_item}, inplace=True)

In [229]:
df['item'].isna().sum()

np.int64(0)

In [230]:
#payment method, location -> 'Unknown'

df.fillna(
    {
        'payment_method': 'Unknown',
        'location': 'Unknown'
    },
    inplace=True
)

In [231]:
# Delete transaction_name where NaT or NaN

before = len(df)
df.dropna(subset = ['transaction_date', 'quantity'], inplace=True)


after = len(df)
print(f'Deleted: {before - after} rows')

Deleted: 914 rows


In [232]:
df[['item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date', 'year', 'month',
       'month_name', 'weekday', 'week']].duplicated().sum()

np.int64(192)

In [233]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [234]:
(df.quantity <= 0).sum()  # 0
(df.price_per_unit <= 0).sum()  # 0
(df.total_spent <= 0).sum()  # 0

np.int64(0)

In [235]:
# df.info()
df.isna().sum()

,0
transaction_id,0
item,0
quantity,0
price_per_unit,48
total_spent,3
payment_method,0
location,0
transaction_date,0
year,0
month,0
